## Module 2 Homework (2026 Cohort)

In this homework, we're going to combine data from various sources to process it in Pandas and generate additional fields.

If not stated otherwise, please use the code snippets covered in the livestream to download and process the data.



In [86]:
# IMPORTS
import numpy as np
import pandas as pd
import requests
from io import StringIO
import re

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt



---
### Question 1: [IPO] Withdrawn IPOs by Company Type

**What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?**

From the Recently Filed IPO list ([iposcoop.com/ipos-recently-filed](https://www.iposcoop.com/ipos-recently-filed/)), collect and process the data to find out which company type saw the most withdrawn IPO value before Sep 11, 2026.

#### Steps:
1. **Data Loading:** Use `pandas.read_html()` with the URL above to load the IPO recently filed table. Filter the rows to keep only those where 'Expected To Trade' is 'Withdrawn'. You should identify 32 entries.
2. **Company Classification:** Create a new column called **Company Type**, categorizing company names based on patterns (order matters, assign the first matched value):
    - "Technologies" -> Technologies
    - "Acquisition Corp", "Acquisition Corporation", or "Corp" -> Acquisition Corp
    - "Inc" or "Incorporated" -> Inc.
    - "Group" -> Group
    - "Ltd" or "Limited" -> Limited
    - "Holdings" or "Holding" -> Holdings
    - Others -> Other

    **Note:** The order of the rules above is important — use the first matching rule. For example, "EUPEC International Group Ltd." will be classified as `Group` (not `Limited`), since the "Group" rule appears before the "Ltd"/"Limited" rule. Also, matches must be exact: "Xinxu Copper Industry Technology Ltd." will be classified as `Limited` (not `Technologies`), because "Technology" does not match the "Technologies" pattern.
3. **Price Parsing:** Define a new field **Avg_price** by parsing the 'Price Low' and 'Price High' fields. Create a function to extract numeric values (e.g., '$8.00' -> 8.0) and calculate the average between low and high. Handle '-' or missing values as `None`/`NaN`.
4. **Numeric Conversion:** Convert 'Shares (millions)' and 'Est \$ Vol (millions)' to numeric formats, cleaning currency symbols (\$) and commas where necessary.
5. **Value Calculation:** Create a new column **Shares_offered_value**:
    - If `Shares (millions) * Avg_price` is not null, use that value.
    - Otherwise, use the value from the `Est $ Vol (millions)` column.
6. **Aggregation:** Group by **Company Type** and calculate the sum of **Shares_offered_value**.

**Answer:** Which class had the highest total value of withdrawals, and what was that value?

In [87]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
}

url = "https://www.iposcoop.com/ipos-recently-filed/"
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

ipo_dfs = pd.read_html(StringIO(response.text))
ipo_dfs

[      File Date                                Company  Symbol  \
 0    2026-09-16           Haymaker Acquisition Corp. V   HYACU   
 1    2026-09-16         Lower Cross Acquisitions Corp.   LCACU   
 2    2026-09-16                     NuvOx Therapeutics    NUOX   
 3    2026-09-16               OceanHawk Acquisition II   OHIIU   
 4    2026-09-15                 Amaero (Cross-Listing)    AMRO   
 ..          ...                                    ...     ...   
 495  2025-12-19      TGE Value Creative Solutions Corp  BEBE.U   
 496  2025-12-17         American Drive Acquisition Co.   ADACU   
 497  2025-12-17                ELC Group Holdings Ltd.    ELCG   
 498  2025-12-17              HAMA Intelligence Limited    HAMA   
 499  2025-12-17  Launchpad Cadenza Acquisition Corp. I   LPCVU   
 
                             Managers  Shares (millions) Price Low Price High  \
 0               Cantor/William Blair              25.00    $10.00     $10.00   
 1                              

In [88]:
df = ipo_dfs[0]
df = df[df['Expected To Trade'] == 'Withdrawn']
print(df.head())
print(df.info())

     File Date                                 Company Symbol  \
8   2026-09-15                  OTSAW Ltd. (Withdrawn)   OTSA   
18  2026-09-10         Motive Technologies (Withdrawn)   MTVE   
24  2026-09-04           Idea Tech Holding (Withdrawn)   IDTL   
34  2026-09-01  Hornbeck Offshore Services (Withdrawn)    HOS   
36  2026-08-31   Coolbit Technologies Ltd. (Withdrawn)   CBAI   

                                             Managers  Shares (millions)  \
8                                       Aegis Capital                4.4   
18  J.P.Morgan/Citigroup/Barclays/Jefferies/RBC Ca...                0.0   
24                                R.F. Lafferty & Co.                2.0   
34  J.P. Morgan/ Barclays/DNB Markets/Piper Sandle...                0.0   
36                               Eddid Securities USA                5.0   

   Price Low Price High Est $ Vol (millions) Expected To Trade SCOOP Rating  
8      $4.50      $5.50               $22.00         Withdrawn          S/

In [89]:
name = df["Company"]

conditions = [
    name.str.contains("Technologies", na=False),
    name.str.contains("Acquisition Corp|Acquisition Corporation|Corp", na=False),
    name.str.contains("Inc|Incorporated", na=False),
    name.str.contains("Group", na=False),
    name.str.contains("Ltd|Limited", na=False),
    name.str.contains("Holdings|Holding", na=False),
]

choices = ["Technologies", "Acquisition Corp", "Inc.", "Group", "Limited", "Holdings"]

df["Company Type"] = np.select(conditions, choices, default="Other")

In [90]:
def parse_price(value):
    """Convert '$8.00' -> 8.0; '-', '', None, NaN -> NaN."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    cleaned = re.sub(r"[^\d.]", "", str(value))  # keep only digits and dots
    if cleaned in ("", "."):
        return np.nan
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

df["Price Low"] = df["Price Low"].apply(parse_price)
df["Price High"] = df["Price High"].apply(parse_price)

df["Avg_price"] = (df["Price Low"] + df["Price High"]) / 2

In [91]:
for col in ["Shares (millions)", "Est $ Vol (millions)"]:
    df[col] = df[col].apply(parse_price)

print(df[["Shares (millions)", "Est $ Vol (millions)"]].dtypes)

Shares (millions)       float64
Est $ Vol (millions)    float64
dtype: object


In [92]:
calculated = df["Shares (millions)"] * df["Avg_price"]
df["Shares_offered_value"] = calculated.fillna(df["Est $ Vol (millions)"])

In [93]:
totals = (
    df.groupby("Company Type")["Shares_offered_value"]
      .sum()
      .sort_values(ascending=False)
)
print(totals)

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             225.8500
Technologies        184.9000
Group                32.5000
Name: Shares_offered_value, dtype: float64


In [94]:
top_class = totals.idxmax()
top_value = totals.max()
print(f"{top_class}: ${top_value:,.2f} million")

Acquisition Corp: $499.99 million


The Acquisition Corp class had the highest total value, $499.99 million. This is usually the name typically used by a SPAC (company with no business of its own) and it exists for raising money from investors and then use that money to buy (merge with) a private company, which makes that private company public.

Inc. are corporations (a legal entity separated from its owners). Shareholders own it, and their personal liability is limited to what they invested.

Holdings: a parent company whose main job is to own other companies (subsidiaries) rather than run operations directly

---
### Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

**What is the median Sharpe ratio (as of 11 September 2026) for companies that went public before 1 September 2025?**

The goal is to replicate the large-scale `yfinance` OHLCV data download and perform basic financial calculations on IPO stocks.

#### Steps:
1.  **Data Loading:** Download the list of 231 IPOs in 2025 from `https://www.iposcoop.com/2025-pricings/`.
2.  **Filtering:** Filter the list to keep only those IPOs with an 'Offer Date' before **1 September 2025**. Also, exclude entries with a 0% return to ensure active tickers are processed. You should see 148 stocks.
3.  **Data Download:** Use `yfinance` to download daily stock data for the filtered tickers and save it to the `stocks_df` dataframe. Make sure you can correctly process the cases when stocks are not present in Yahoo Finance (probably delisted). At this stage you should see about 134 stocks.
4.  **Feature Engineering:**
    *   Define `growth_252d` as `Close / Close.shift(252)` to represent growth after approximately one year of trading.
    *   Calculate **annualized volatility** using the specific formula: `stocks_df['volatility'] = stocks_df['Close'].rolling(30).std() * np.sqrt(252)`.
5.  **Sharpe Ratio Calculation:** Calculate the Sharpe ratio assuming a risk-free rate of **5.0%** (0.05) - it is close to the current value of the US 10Y Treasury bond yield:
    *   `stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']`
6.  **Final Analysis:** Filter the resulting DataFrame to keep data only for the trading day **'2026-09-11'** and compute descriptive statistics (using the `describe()` function).

#### Expected Observations:
*   Compare the median vs. mean for `growth_252d` to see if high-growth outliers are biasing the average.
*   Identify the count of stocks that successfully reached the 252-day trading milestone.
*   Do you see examples of stocks with risk-adjusted returns (Sharpe ratio) more attractive than pure growth statistics?

**Answer:** What is the median Sharpe ratio for these stocks as of September 11, 2026?

In [95]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
}

url = "https://www.iposcoop.com/2025-pricings/"
response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

ipos_2025 = pd.read_html(StringIO(response.text))[0]   # a DataFrame

print(type(ipos_2025))   # should say <class 'pandas.core.frame.DataFrame'>
print(len(ipos_2025))    # should be 231

<class 'pandas.DataFrame'>
231


In [96]:
df = ipos_2025.copy()

# Convert dates; unparseable values become NaT
df["Offer Date"] = pd.to_datetime(df["Offer Date"], errors="coerce")

# Convert '25.50%' -> 25.5; '-' or blanks become NaN
df["Return"] = pd.to_numeric(
    df["Return"].astype(str).str.replace(r"[%,\s]", "", regex=True),
    errors="coerce",
)

filtered = df[
    (df["Offer Date"] < "2025-09-01") &
    (df["Return"] != 0)
].copy()

filtered.head()

,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00,S/O
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00,S/O
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75,S/O
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,$4.00,$4.02,$0.37,-90.75,S/O
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,$10.00,$10.12,$10.12,1.20,S/O


In [97]:
import yfinance as yf

# Tickers from the filtered IPO list (step 2)
ALL_TICKERS = filtered["Symbol"].dropna().astype(str).str.strip().unique().tolist()

stocks_df = pd.DataFrame({'A': []})
failed = []

for i, ticker in enumerate(ALL_TICKERS):
    print(i, ticker)

    # Work with stock prices
    historyPrices = yf.download(tickers=ticker,
                                period="max",
                                interval="1d",
                                multi_level_index=False,
                                progress=False)

    # Delisted / not on Yahoo Finance -> empty dataframe, skip it
    if historyPrices.empty:
        failed.append(ticker)
        continue

    # generate features for historical prices
    historyPrices['Ticker'] = ticker
    historyPrices['Year'] = historyPrices.index.year
    historyPrices['Month'] = historyPrices.index.month
    historyPrices['Weekday'] = historyPrices.index.weekday
    historyPrices['Date'] = historyPrices.index.date

    # Step 4: growth after ~1 year and 30d rolling volatility
    historyPrices['growth_252d'] = historyPrices['Close'] / historyPrices['Close'].shift(252)
    historyPrices['volatility'] = historyPrices['Close'].rolling(30).std() * np.sqrt(252)

    # Step 5: Sharpe ratio with a 5% risk-free rate
    historyPrices['Sharpe'] = (historyPrices['growth_252d'] - 0.05) / historyPrices['volatility']

    # sleep between downloads - not to overload the API server
    time.sleep(1)

    if stocks_df.empty:
        stocks_df = historyPrices
    else:
        stocks_df = pd.concat([stocks_df, historyPrices], ignore_index=True)

print("Downloaded:", stocks_df['Ticker'].nunique())   # about 134
print("Not found:", len(failed), failed)

0 PMI
1 TGHL
2 THH
3 CURX
4 CEPF
5 ETS
6 YMT
7 NUTR
8 PPCB
9 BUUU
10 MIAX
11 RYOJ
12 BLSH
13 NSRX
14 MAGH
15 AVBH
16 DKI
17 EFTY
18 HTFL
19 FLY
20 WYFI
21 CTW
22 HCMAU
23 FIG
24 SI
25 AMBQ
26 YMAT
27 ARX
28 LHAI
29 MH
30 CARL
31 CRE
32 TDIC
33 NIQ


$MJID: possibly delisted; no timezone found

1 Failed download:
['MJID']: possibly delisted; no timezone found


34 MJID
35 KMRK
36 MGRT
37 ALM
38 DLXY
39 MSGY
40 ANPA
41 TLIH
42 MAMK
43 CV


$EMPG: possibly delisted; no timezone found

1 Failed download:
['EMPG']: possibly delisted; no timezone found


44 EMPG
45 GRAN


...

121 EPWK
122 FBGL
123 CJMB
124 HCAI
125 PLUT
126 INR
127 MAZE


$MTSR: possibly delisted; no timezone found

1 Failed download:
['MTSR']: possibly delisted; no timezone found


128 MTSR
129 BBNX
130 SFD
131 AAPG
132 VG


$SKBL: possibly delisted; no timezone found

1 Failed download:
['SKBL']: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found

1 Failed download:
['MCTR']: possibly delisted; no timezone found


133 SKBL
134 MCTR
135 DGNX
136 TOPP
137 FLOC
138 PCLA
139 HKPD
140 UFG
141 MIMI
142 MASK
143 CEPO
144 ZYBT
145 INLF
Downloaded: 133
Not found: 13 ['MJID', 'EMPG', 'CAEP', 'PTNM', 'AHL', 'CEPT', 'SDM', 'TBH', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


In [98]:
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
final_df = stocks_df[stocks_df.Date == '2026-09-11']
final_df.head()

,Close,High,Low,Open,Volume,Ticker,Year,Month,Weekday,Date,growth_252d,volatility,Sharpe
259,5.330,5.330,4.550,4.560,523200,PMI,2026,9,4,2026-09-11,0.016451,32.642574,-0.001028
524,0.947,0.971,0.913,0.950,196200,TGHL,2026,9,4,2026-09-11,0.444601,1.921418,0.205370
790,1.670,1.870,1.600,1.831,33100,THH,2026,9,4,2026-09-11,0.029298,6.093873,-0.003397
1057,4.550,4.550,4.327,4.340,7400,CURX,2026,9,4,2026-09-11,0.029242,6.245544,-0.003324
1327,10.355,10.375,10.350,10.350,5600,CEPF,2026,9,4,2026-09-11,1.024233,0.351583,2.770994


In [99]:
stats = final_df[['growth_252d', 'volatility', 'Sharpe']].describe()
print(stats)

       growth_252d  volatility      Sharpe
count   130.000000  131.000000  130.000000
mean      1.057613   27.517135         inf
std       3.049007   66.882596         NaN
min       0.001005    0.000000   -0.040147
25%       0.141035    2.301309    0.012042
50%       0.594534    7.885748    0.050142
75%       1.038821   23.452319    0.133619
max      33.638270  627.749268         inf


c:\Users\rfern\Desktop\python-projects\stock-markets-analytics-zoomcamp\.venv\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Growth: How much the investment gained or lost. If you bought the stock a year ago, what is your money worth now? There is 252 because markets are closed on weekends and holidays, so a year has about 252 trading days, not 365. For IPOs, one-year growth is a classic question: companies often come to market with lots of excitement, and investors want to know whether buying at that time actually paid off.
growth = new_price/old_price

Volatility: how bumpy the path was. Two stocks can end the year at the same place, but get there very differently. In finance, volatility is the standard way to measure risk. High volatility means the price is unpredictable, so you could end up far above or far below where you expected.

Sharpe ratio: Was the risk worth it? The sharpe ratio combines the ideas above into one number: Sharpe = (return - safe return) / volatility. For each unit of risk I took, how much extra return did I get?

return = (new_price-old_price)/old_price

In [100]:
# 1) Mean vs median growth: is the average pulled up by outliers?
print("growth_252d mean:  ", final_df['growth_252d'].mean())
print("growth_252d median:", final_df['growth_252d'].median())

growth_252d mean:   1.0576134223609819
growth_252d median: 0.5945340683779607


The median tells us that the middle stock kept only 59% of its value, a loss about 41%.
A mean growth of 1.06 suggests a gain about 6%.

This indicates that the mean is pulled up by a few huge winners.

In [101]:
# 2) How many stocks reached 252 trading days?
print("Stocks with growth_252d:", final_df['growth_252d'].notna().sum())

Stocks with growth_252d: 130


In [102]:
# 3) Sharpe vs pure growth: compare the rankings
ranked = final_df[['Ticker', 'growth_252d', 'volatility', 'Sharpe']].dropna()
print("\nTop by growth:\n", ranked.sort_values('growth_252d', ascending=False).head(10))
print("\nTop by Sharpe:\n", ranked.sort_values('Sharpe', ascending=False).head(10))


Top by growth:
       Ticker  growth_252d  volatility        Sharpe
18524   MGRT    33.638270  131.115959  2.561722e-01
23367   MAMK     6.532663    0.000000           inf
5836    BUUU     5.930931   69.792390  8.426321e-02
7215    MAGH     5.451613    0.000001  4.195710e+06
21857    ALM     3.480899   36.399039  9.425795e-02
57743   XHLD     2.449658   52.991098  4.528417e-02
6386    RYOJ     2.268293    7.197246  3.082141e-01
13355   EFTY     2.106592    0.000000           inf
25847   SLDE     1.852848   24.497550  7.359298e-02
23671     CV     1.818182    6.754878  2.617637e-01

Top by Sharpe:
       Ticker  growth_252d  volatility        Sharpe
23367   MAMK     6.532663    0.000000           inf
13355   EFTY     2.106592    0.000000           inf
7215    MAGH     5.451613    0.000001  4.195710e+06
14757  HCMAU     1.034585    0.182827  5.385326e+00
1327    CEPF     1.024233    0.351583  2.770994e+00
31825   TMDE     0.776166    0.414673  1.751178e+00
23064   TLIH     0.989673    0

This shows that there are three different type of stocks.
* Frozen stocks: volatility of 0. A real traded stock can't sit at exactly the same price for six weeks. These stocks were most likely halted, suspended or delisted, and Yahoo Finance keeps repeating the last known price. Their sharpe rations (inf and 4 million) are not real results, only division by zero.
* SPACs: steady prices just above 10$. SPACs go public at 10$ per share, and the money sits in a trust account that earns interest, mostly from US TReasury bills. So the price creeps slowly above 10$ and barely moves otherwise. That gives very low volatility and a small, steady gain, which is exactly why they have high Sharpe ratios.
* Penny stocks: prices below 1$. These stocks lost a lot of value, yet they're in the sharpe top 10. This confirms the price-level problem. A stock trading 0.47$ can only move by a few cents a day, so its volatility measured in dollars is tiny. Divide by a tiny number, and the Sharpe ratio looks right.

In general, bigger growth came with bigger volatility. 

In [103]:
# ANSWER
print("\nMedian Sharpe ratio on 2026-09-11:", round(final_df['Sharpe'].median(), 4))


Median Sharpe ratio on 2026-09-11: 0.0501


---
### Question 3: [IPO] 'Fixed Months Holding Strategy'

**What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize the median growth value?**

#### Goal:
Investigate the performance of 2025 IPO stocks over fixed time horizons (1 to 12 months) to identify the holding period that yields the highest median return relative to the first day's closing price.

#### Steps:
1. **Data Source:** Use the existing `stocks_df` containing daily OHLCV data and calculated features for the 2025 IPOs filtered in Question 2.
2. **Feature Engineering:** Calculate 12 future growth columns representing fixed holding periods:
   - `future_growth_1_m`, `future_growth_2_m`, ..., `future_growth_12_m`.
   - Assume 1 month equals 21 trading days (e.g., 1 month = 21 days, 2 months = 42 days, ..., 12 months = 252 days).
3. **Identify Entry Points:** For each ticker, determine the first available trading day (`min_date`) and the corresponding closing price.
4. **Data Alignment:** Perform an inner join between the `min_date` records and the full growth dataset. This isolates the returns for each stock starting specifically from its IPO date.
5. **Median Analysis:** Compute descriptive statistics for these 12 columns. Specifically, identify the month where the **50th percentile (median)** value is highest.

#### Observations:
- Do you observe any trend of the median growth over time? What does it mean for an investor eager to base their strategy on the newly IPOed companies?
- Compare the median to the mean to understand how extreme outliers (high-growth stocks) affect the average versus the typical stock's performance.

**Answer:** Based on the analysis of the filtered tickers, what is the optimal holding period (in months) and its corresponding maximum median growth value?

In [104]:
stocks_df = stocks_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

for m in range(1, 13):
    days = 21 * m
    stocks_df[f'future_growth_{m}_m'] = (
        stocks_df.groupby('Ticker')['Close'].shift(-days) / stocks_df['Close']
    )

In [105]:
min_dates = stocks_df.groupby('Ticker')['Date'].min().reset_index()
min_dates.columns = ['Ticker', 'min_date']
print(min_dates.head())

  Ticker   min_date
0   AAPG 2025-01-24
1   AARD 2025-02-13
2   ADVB 2025-03-06
3    AII 2025-05-08
4   AIRO 2025-06-13


In [106]:
growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]

ipo_first_day = pd.merge(
    stocks_df,
    min_dates,
    how='inner',
    left_on=['Ticker', 'Date'],
    right_on=['Ticker', 'min_date'],
)

print(len(ipo_first_day), "stocks")   # one row per ticker

133 stocks


In [107]:
stats = ipo_first_day[growth_cols].describe()
print(stats.loc[['count', 'mean', '50%']].T)

medians = ipo_first_day[growth_cols].median()
best_month = medians.idxmax()
print("\nMedians:\n", medians)
print(f"\nBest: {best_month} with median growth {medians.max():.4f}")

                    count      mean       50%
future_growth_1_m   132.0  1.119635  0.935351
future_growth_2_m   131.0  1.294861  0.890411
future_growth_3_m   131.0  1.252014  0.827160
future_growth_4_m   131.0  1.135939  0.741678
future_growth_5_m   131.0  1.084528  0.667857
future_growth_6_m   131.0  1.114789  0.726000
future_growth_7_m   131.0  1.146371  0.662000
future_growth_8_m   131.0  1.004473  0.603508
future_growth_9_m   131.0  1.158479  0.580294
future_growth_10_m  131.0  1.185497  0.533333
future_growth_11_m  131.0  1.068597  0.492453
future_growth_12_m  130.0  0.963117  0.529827

Medians:
 future_growth_1_m     0.935351
future_growth_2_m     0.890411
future_growth_3_m     0.827160
future_growth_4_m     0.741678
future_growth_5_m     0.667857
future_growth_6_m     0.726000
future_growth_7_m     0.662000
future_growth_8_m     0.603508
future_growth_9_m     0.580294
future_growth_10_m    0.533333
future_growth_11_m    0.492453
future_growth_12_m    0.529827
dtype: float64

Bes

The medians fall monotonically as holding period lengthens. Buying at the first day's close and holding a year left the typical investor with about half their money. The small upticks at months 6 and 12 are minor wobbles in an otherwise consistent decline.

This is usual, the first-day closing price frequently reflects that excitement rather than the company's underlying value.

---
### Question 4: [Strategy] Simple RSI-Based Trading Strategy

**What is the total profit (in $ thousands) you would have earned by investing $1000 every time a stock was oversold (RSI < 30)?**

#### Goal:
Apply a simple rule-based trading strategy using the Relative Strength Index (RSI) technical indicator to identify oversold signals and calculate cumulative profits over a 25-year period.

#### Steps:
1.  **Data Acquisition:** Use the provided brotli-compressed parquet file containing precomputed technical and macro indicators for a broad set of tickers.
    ```python
    import gdown
    import pandas as pd
    file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
    gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
    df = pd.read_parquet("data.parquet", engine="pyarrow")
    ```
2.  **Strategy Setup:** Define the RSI threshold for an "oversold" signal as **RSI < 30**.
3.  **Filtering:** Filter the dataset to isolate trades occurring between **2000-01-01** and **2025-06-01** where the RSI signal was triggered.
4.  **Profit Calculation:**
    - Assume an investment of **$1,000** for every signal.
    - Use the **30-day forward return** (`growth_future_30d`) to determine the outcome of each trade.
    - Calculate Net Income: `net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()`

#### Observations:
- Increasing the threshold from 25 (we used last year) to 30 significantly increases the number of trading opportunities (from ~1,568 to 5,206).
- With an average 30-day return of **1.26%** and a win rate of **55.13%**, the strategy shows consistent, albeit modest, capital growth over the long term.

**Answer:** Based on a few thousands of trades, what is the net income earned (in $ thousands)?

---

In [1]:
import gdown
import pandas as pd

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)

df = pd.read_parquet("data.parquet", engine="pyarrow")

print(df.shape)
print([c for c in df.columns if 'rsi' in c.lower() or 'growth_future' in c])

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=efb3e845-4efa-48b9-9815-a60e3aed96e0
To: c:\Users\rfern\Desktop\python-projects\stock-markets-analytics-zoomcamp\homework\data.parquet
100%|██████████| 130M/130M [00:24<00:00, 5.25MB/s] 


(229932, 203)
['growth_future_30d', 'rsi', 'fastk_rsi', 'fastd_rsi', 'cdl3starsinsouth']


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

selected_df = df[
    (df['rsi'] < 30) &
    (df['Date'] >= '2000-01-01') &
    (df['Date'] < '2025-06-01')
]

print("Number of trades:", len(selected_df))

Number of trades: 5206


In [3]:
net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()

print(f"Net income: ${net_income:,.2f}")
print(f"In thousands: ${net_income/1000:,.1f}k")

Net income: $65,805.59
In thousands: $65.8k


In [5]:
returns = selected_df['growth_future_30d'] - 1

print("Trades:       ", len(selected_df))
print("Avg return:    {:.2%}".format(returns.mean()))
print("Win rate:      {:.2%}".format((returns > 0).mean()))
print("Median return: {:.2%}".format(returns.median()))
print("Best / worst:  {:.2%} / {:.2%}".format(returns.max(), returns.min()))

Trades:        5206
Avg return:    1.26%
Win rate:      55.13%
Median return: 0.63%
Best / worst:  201.89% / -44.62%


RSI (Relative Strength Index) is a technical indicator between 0 and 100 that measures wether a stock has been rising or falling hard recently. Traders conventionally read below 30 as "oversold" (the price has dropped sharply and may be due for a bounce) and above 70 as "overbought".

The strategy is mechanical: every time any stock in the dataset closes with RSI under 30, buy 1000$ of it, hold exactly 30 days, and sell. No judgment, no stock picking. growth_future_30d tells you what the price did over those 30 days, which is why the profit is a simple sum.

This trategy has 55.13% win rate, meaning that slightly more than half the trades made money, and an average return of 1.26% per trade is small but positive.

### Q5. [Exploratory, Optional] Predicting a Positive-Return IPO

Most of the strategies for investing in IPOs deliver **negative average and median returns** (and even the 75th percentile).

**Question:**
How would you change the strategy if you want to **increase profitability**?

> This is an open-ended brainstorming question — propose ideas for identifying IPOs with positive future returns or building a more effective trading strategy.


IPO returns are very uneven: the median 2025 IPO lost about 47% over a year, while a handful multiplied several times and dragged the mean back to roughly break-even. So a strategy has to either catch those few winners or avoid the many losers.

First, I would use information the price data doesn't contain: sector, size of the offering, whether the company is profitable, whether it is a SPAC, and country of origin. That could feed a model predicting which IPOs end the year up, or just rule out categories that have done badly.

Second, I would filter out the problem cases I hit in the data: stocks under $1, stocks that barely trade and sit frozen for weeks, and very small offerings.

Third, I would change the timing. Question 3 showed the longer you hold, the worse the typical stock does. So rather than buying on day one and waiting, I would test buying around six months in, after insiders are free to sell and that selling has passed, with a capped holding period or a stop-loss.

The catch is that betting on a few picks means a bad selection rule loses everything, not just underperforms. I would test any of this against the 2023 and 2024 IPOs first.